Parse any file from notebooks/logs and convert them to better view the raw data and also group them by gameIDs



In [2]:
# file_name = "raven_events_20251128_131751.jsonl"

file_name = "raven_events_20251128_115613.jsonl"

In [3]:
from __future__ import annotations

from collections import defaultdict
import json
from pathlib import Path
from typing import Iterable


logs_dir = Path("notebooks/logs")
input_file = logs_dir / file_name
output_file = input_file.with_suffix(".json")


def iter_jsonl(path: Path) -> Iterable[dict]:
    """Yield each JSON object stored per-line in a .jsonl file."""
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            record = json.loads(line)
            raw_value = record.get("raw")
            if isinstance(raw_value, str):
                try:
                    record["raw"] = json.loads(raw_value)
                except json.JSONDecodeError:
                    # keep original string if it isn't valid JSON
                    pass
            yield record


def jsonl_to_grouped_json(src: Path, dest: Path | None = None) -> Path:
    """Convert JSONL into a JSON dict grouped by gameId with seq order preserved."""
    if dest is None:
        dest = src.with_suffix(".json")

    events = list(iter_jsonl(src))
    events.sort(key=lambda record: record.get("seq", 0))

    grouped: dict[str, list[dict]] = defaultdict(list)
    for event in events:
        raw = event.get("raw") if isinstance(event.get("raw"), dict) else {}
        game_id = event.get("gameId") or raw.get("gameId") or "unknown"
        grouped[game_id].append(event)

    dest.write_text(json.dumps(grouped, indent=2), encoding="utf-8")
    return dest


written_path = jsonl_to_grouped_json(input_file, output_file)
print(f"Wrote {written_path}")



Wrote notebooks\logs\raven_events_20251128_115613.json
